# Meta Thinker: Testing Reasoning Strategies

This notebook demonstrates testing and evaluation of Meta Thinker's reasoning capabilities across diverse problem domains:

1. **Game24**: Arithmetic puzzle solving
2. **Sudoku**: Constraint satisfaction problems
3. **PIQA**: Physical commonsense reasoning
4. **PubMedQA**: Biomedical question answering
5. **GPQA**: Graduate-level science questions
6. **OpenBookQA**: Elementary science questions
7. **OR-Instruct**: Operations research problems

## Purpose

- Verify that Meta Thinker works across different problem types
- Compare reasoning quality across domains
- Identify strengths and weaknesses of different strategies
- Demonstrate versatility of the meta-reasoning approach

## Prerequisites

- Completed notebooks 00 (data loading) and 01 (meta-learning)
- API keys configured for GPT-4 or DeepSeek
- Utils modules from the project

## Setup and Imports

In [ ]:
from utils.utils import *
from utils.agent import *
from datasets import load_dataset
import json

print("✓ Imports successful")

## Helper Functions

In [ ]:
def test_reasoning(client, prompt: str, model_type: str = 'gpt') -> str:
    """
    Test reasoning on a given prompt.
    
    Args:
        client: LLM client
        prompt: The problem/question to solve
        model_type: Type of model ('gpt' or 'deepseek')
    
    Returns:
        Model's response
    """
    messages = [{'role': 'user', 'content': prompt}]
    return pipeline_message(messages, client)


def extract_answer(response: str) -> str:
    """
    Extract the answer from a structured response.
    
    Args:
        response: Full response with <answer> tags
    
    Returns:
        Extracted answer text
    """
    if '<answer>' in response and '</answer>' in response:
        start = response.find('<answer>') + len('<answer>')
        end = response.find('</answer>')
        return response[start:end].strip()
    return response


print("✓ Helper functions defined")

## 1. Testing on Game24 Puzzle

The 24-point game requires using four numbers with arithmetic operations to reach 24.

In [ ]:
# Initialize client
client = initialize_pipeline('gpt')

# Define Game24 problem
game24_prompt = """You are a task solver for the 24-point game.

Rules:
- Use each given number exactly once
- Use only basic arithmetic operators: +, -, *, /
- Combine numbers to obtain exactly 24

Given numbers: 1, 2, 3, 4

Let's think step by step.

Format your response as:
<think>
[Step-by-step reasoning]
</think>

<answer>
[Final equation that equals 24]
</answer>
"""

# Test
print("Testing Game24 with numbers 1, 2, 3, 4...\n")
game24_response = test_reasoning(client, game24_prompt)

print("=" * 70)
print("GAME24 RESPONSE:")
print("=" * 70)
print(game24_response)
print("=" * 70)

# Extract answer
game24_answer = extract_answer(game24_response)
print(f"\n✓ Extracted Answer: {game24_answer}")

## 2. Testing on Sudoku Puzzle

Sudoku requires constraint satisfaction and logical deduction.

In [ ]:
sudoku_prompt = """You are an expert puzzle solver. Solve this 4×4 Sudoku puzzle step-by-step.

Rules:
- Fill the grid with numbers 1 to 4
- Each row must contain all numbers 1 to 4 with no repeats
- Each column must contain all numbers 1 to 4 with no repeats
- Each 2×2 subgrid must contain all numbers 1 to 4 with no repeats

Puzzle (0 represents empty cells):
[
  [0, 2, 0, 4],
  [0, 0, 3, 0],
  [0, 1, 0, 0],
  [4, 0, 0, 0]
]

Format your response as:
<think>
[Step-by-step reasoning with constraint analysis]
</think>

<answer>
[Complete solved 4x4 grid]
</answer>
"""

print("Testing Sudoku solving...\n")
sudoku_response = test_reasoning(client, sudoku_prompt)

print("=" * 70)
print("SUDOKU RESPONSE:")
print("=" * 70)
print(sudoku_response[:500] + "...\n" if len(sudoku_response) > 500 else sudoku_response + "\n")
print("=" * 70)

sudoku_answer = extract_answer(sudoku_response)
print(f"\n✓ Extracted Solution:\n{sudoku_answer}")

## 3. Testing on PIQA (Physical Interaction QA)

PIQA tests physical commonsense reasoning about everyday situations.

In [ ]:
# Load PIQA dataset
print("Loading PIQA dataset...")
piqa_dataset = load_dataset("piqa", trust_remote_code=True)
piqa_train = piqa_dataset['train']

print(f"✓ Loaded PIQA: {len(piqa_train)} training examples")

# Display dataset structure
print("\nDataset structure:")
print(f"Features: {piqa_train.features}")
print(f"\nExample question:")
example = piqa_train[0]
print(f"Goal: {example['goal']}")
print(f"Solution 1: {example['sol1']}")
print(f"Solution 2: {example['sol2']}")
print(f"Correct label: {example['label']}")

In [ ]:
# Test on a PIQA example
piqa_example = piqa_train[10]

piqa_prompt = f"""You need to choose the better solution for achieving a goal.

Goal: {piqa_example['goal']}

Solution 1: {piqa_example['sol1']}
Solution 2: {piqa_example['sol2']}

Which solution is better and why?

Format your response as:
<think>
[Analyze both solutions and explain your reasoning]
</think>

<answer>
[Solution 1 or Solution 2, with brief justification]
</answer>
"""

print("Testing PIQA reasoning...\n")
piqa_response = test_reasoning(client, piqa_prompt)

print("=" * 70)
print("PIQA RESPONSE:")
print("=" * 70)
print(piqa_response)
print("=" * 70)

piqa_answer = extract_answer(piqa_response)
correct_label = "Solution 1" if piqa_example['label'] == 0 else "Solution 2"
print(f"\n✓ Model's answer: {piqa_answer}")
print(f"✓ Correct answer: {correct_label}")

## 4. Testing on PubMedQA

PubMedQA tests biomedical reasoning based on research abstracts.

In [ ]:
# Load PubMedQA dataset
print("Loading PubMedQA dataset...")
pubmed_dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
pubmed_train = pubmed_dataset['train']

print(f"✓ Loaded PubMedQA: {len(pubmed_train)} training examples")

# Display example
pubmed_example = pubmed_train[10]
print("\nExample question:")
print(f"Question: {pubmed_example['question']}")
print(f"\nContext (first 200 chars): {pubmed_example['context']['contexts'][0][:200]}...")
print(f"\nFinal decision: {pubmed_example['final_decision']}")

In [ ]:
# Test on PubMedQA
pubmed_test = pubmed_train[5]

pubmed_prompt = f"""You are analyzing a biomedical research question.

Question: {pubmed_test['question']}

Context: {pubmed_test['context']['contexts'][0]}

Based on the context, answer the question with "yes", "no", or "maybe".

Format your response as:
<think>
[Analyze the evidence from the context]
</think>

<answer>
[yes/no/maybe with brief explanation]
</answer>
"""

print("Testing PubMedQA reasoning...\n")
pubmed_response = test_reasoning(client, pubmed_prompt)

print("=" * 70)
print("PUBMEDQA RESPONSE:")
print("=" * 70)
print(pubmed_response[:600] + "...\n" if len(pubmed_response) > 600 else pubmed_response + "\n")
print("=" * 70)

pubmed_answer = extract_answer(pubmed_response)
print(f"\n✓ Model's answer: {pubmed_answer[:100]}...")
print(f"✓ Correct answer: {pubmed_test['final_decision']}")
print(f"✓ Reference explanation: {pubmed_test['long_answer'][:200]}...")

## 5. Testing on GPQA (Graduate-Level Science)

GPQA contains challenging graduate-level questions in physics, chemistry, and biology.

In [ ]:
# Load GPQA dataset
print("Loading GPQA dataset...")
try:
    gpqa_dataset = load_dataset("Idavidrein/gpqa", "gpqa_diamond")
    gpqa_train = gpqa_dataset['train']
    
    print(f"✓ Loaded GPQA: {len(gpqa_train)} examples")
    
    # Display structure
    print("\nKey fields available:")
    key_fields = ['Question', 'Correct Answer', 'Incorrect Answer 1', 
                  'Incorrect Answer 2', 'Incorrect Answer 3', 'Explanation']
    for field in key_fields:
        if field in gpqa_train[0]:
            print(f"  - {field}")
    
    # Show example
    print("\nExample question:")
    print(f"Question: {gpqa_train[0]['Question'][:200]}...")
    print(f"Domain: {gpqa_train[0].get('High-level domain', 'N/A')}")
    
except Exception as e:
    print(f"⚠ Note: GPQA requires authentication. Error: {e}")
    print("You may need to login with: huggingface-cli login")

## 6. Testing on OpenBookQA

OpenBookQA tests elementary science knowledge.

In [ ]:
# Load OpenBookQA dataset
print("Loading OpenBookQA dataset...")
openbookqa_dataset = load_dataset("allenai/openbookqa", "main")
openbookqa_train = openbookqa_dataset['train']

print(f"✓ Loaded OpenBookQA: {len(openbookqa_train)} training examples")

# Display example
openbookqa_example = openbookqa_train[0]
print("\nExample question:")
print(f"Question: {openbookqa_example['question_stem']}")
print(f"\nChoices:")
for label, text in zip(openbookqa_example['choices']['label'], 
                       openbookqa_example['choices']['text']):
    print(f"  {label}: {text}")
print(f"\nCorrect answer: {openbookqa_example['answerKey']}")

In [ ]:
# Test on OpenBookQA
obqa_test = openbookqa_train[5]

obqa_prompt = f"""Answer this elementary science question.

Question: {obqa_test['question_stem']}

Choices:
"""

for label, text in zip(obqa_test['choices']['label'], obqa_test['choices']['text']):
    obqa_prompt += f"  {label}: {text}\n"

obqa_prompt += """
Format your response as:
<think>
[Explain your reasoning]
</think>

<answer>
[Letter of correct choice with brief explanation]
</answer>
"""

print("Testing OpenBookQA reasoning...\n")
obqa_response = test_reasoning(client, obqa_prompt)

print("=" * 70)
print("OPENBOOKQA RESPONSE:")
print("=" * 70)
print(obqa_response)
print("=" * 70)

obqa_answer = extract_answer(obqa_response)
print(f"\n✓ Model's answer: {obqa_answer}")
print(f"✓ Correct answer: {obqa_test['answerKey']}")

## 7. Testing on OR-Instruct (Operations Research)

OR-Instruct contains operations research and optimization problems.

In [ ]:
# Load OR-Instruct dataset
print("Loading OR-Instruct dataset...")
or_dataset = load_dataset("CardinalOperations/OR-Instruct-Data-3K")
or_train = or_dataset['train']

print(f"✓ Loaded OR-Instruct: {len(or_train)} examples")

# Display example
or_example = or_train[0]
print("\nExample problem:")
print(f"Prompt: {or_example['prompt'][:300]}...")
print(f"\nSolution preview: {or_example['completion'][:200]}...")

In [ ]:
# Test on OR-Instruct
or_test = or_train[2]

or_prompt = f"""{or_test['prompt']}

Solve this operations research problem step-by-step.

Format your response as:
<think>
[Break down the problem and explain your approach]
</think>

<answer>
[Provide the solution]
</answer>
"""

print("Testing OR-Instruct reasoning...\n")
or_response = test_reasoning(client, or_prompt)

print("=" * 70)
print("OR-INSTRUCT RESPONSE:")
print("=" * 70)
print(or_response[:600] + "...\n" if len(or_response) > 600 else or_response + "\n")
print("=" * 70)

or_answer = extract_answer(or_response)
print(f"\n✓ Model's solution preview: {or_answer[:200]}...")
print(f"\n✓ Reference solution preview: {or_test['completion'][:200]}...")

## Summary: Cross-Domain Testing Results

### Datasets Tested:

| Dataset | Domain | Difficulty | Question Type |
|---------|--------|------------|---------------|
| **Game24** | Arithmetic | Medium | Puzzle solving |
| **Sudoku** | Logic | Medium | Constraint satisfaction |
| **PIQA** | Physical reasoning | Easy-Medium | Multiple choice |
| **PubMedQA** | Biomedical | Hard | Yes/No/Maybe |
| **GPQA** | Graduate science | Very Hard | Multiple choice |
| **OpenBookQA** | Elementary science | Easy | Multiple choice |
| **OR-Instruct** | Operations research | Hard | Open-ended |

### Key Observations:

1. **Structured Reasoning**: The `<think>` and `<answer>` format helps organize thoughts across all domains

2. **Domain Versatility**: Meta Thinker adapts its reasoning style to different problem types

3. **Complexity Handling**: Performance varies with domain difficulty, as expected

4. **Format Consistency**: Structured output makes it easy to extract and evaluate answers

### Next Steps:

- Run quantitative evaluation across larger samples
- Compare meta-reasoning vs fixed strategies systematically  
- Analyze failure cases to improve the approach
- Fine-tune models specifically for meta-reasoning

### Related Files:

- `meta_learn.py`: Script for systematic meta-learning experiments
- `main.py`: Batch data generation script
- `00_setup_and_data.ipynb`: Data loading tutorial
- `01_meta_learning_demo.ipynb`: Meta-learning demonstrations

## Optional: Batch Testing Function

For systematic evaluation across multiple examples.

In [ ]:
def batch_test_dataset(client, dataset, dataset_name: str, 
                       n_samples: int = 10, format_fn=None):
    """
    Test reasoning on multiple samples from a dataset.
    
    Args:
        client: LLM client
        dataset: Dataset to test on
        dataset_name: Name of the dataset
        n_samples: Number of samples to test
        format_fn: Function to format examples into prompts
    
    Returns:
        List of results
    """
    results = []
    
    print(f"\nBatch testing on {dataset_name}...")
    print(f"Testing {n_samples} samples")
    
    for i in range(min(n_samples, len(dataset))):
        try:
            # Format prompt
            if format_fn:
                prompt = format_fn(dataset[i])
            else:
                prompt = str(dataset[i])
            
            # Get response
            response = test_reasoning(client, prompt)
            
            # Store result
            results.append({
                'index': i,
                'prompt': prompt,
                'response': response,
                'answer': extract_answer(response)
            })
            
            if (i + 1) % 5 == 0:
                print(f"  Progress: {i + 1}/{n_samples}")
        
        except Exception as e:
            print(f"  Error on sample {i}: {e}")
            continue
    
    print(f"✓ Completed {len(results)} samples")
    return results


# Example usage:
# game24_results = batch_test_dataset(client, game24_dataset, "Game24", n_samples=20)

print("✓ Batch testing function defined")